In [20]:
# Notebook used to train and save models I found it was easiest to have all of the models and dataset functions in this script to make sure everything worked.

from os import write
import random
import csv
import pickle

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import pandas as pd
from matplotlib import pyplot as plt
import numpy as np

!pip install torchinfo
from torchinfo import summary


In [4]:
url = "https://raw.githubusercontent.com/lledson425/Machine-Learning-Final-Project/ee32c077f2fba84565378129f5a017e7a75b8de6/data/train_algebra_dataset.csv"
train_df = pd.read_csv(url)
train_df.iloc[20:26]

,type,input_text,target_text
20,quadratic,x^2 - 2x - 35 = 0 ->,(x + 5)(x - 7) = 0
21,linear,3x + 9 = 33 ->,3x = 24
22,linear,3x = 24 ->,x= 8
23,linear,9x + 3 = 93 ->,9x = 90
24,linear,9x = 90 ->,x= 10
25,quadratic,x^2 - 3x - 18 = 0 ->,(x + 3)(x - 6) = 0


In [5]:
def algebra_splitter(text):
    text = str(text).strip()
    tokens = []
    i = 0

    while i < len(text):
        ch = text[i]

        # handle arrow "->"
        if ch == '-' and i + 1 < len(text) and text[i + 1] == '>':
            tokens.append("->")
            i += 2
            continue

        # skip spaces
        if ch.isspace():
            i += 1
            continue

        # numbers (multi-digit)
        if ch.isdigit():
            num = ch
            i += 1
            while i < len(text) and text[i].isdigit():
                num += text[i]
                i += 1
            tokens.append(num)
            continue

        # variables (like x)
        if ch.isalpha():
            tokens.append(ch)
            i += 1
            continue

        # operators / parentheses
        if ch in "+-*/^=()":
            tokens.append(ch)
            i += 1
            continue

        # fallback (rare cases)
        tokens.append(ch)
        i += 1

    # add in explicit multiplication symbols
    new_tokens = []
    for j in range(len(tokens)):
        new_tokens.append(tokens[j])

        if j < len(tokens) - 1:
            a = tokens[j]
            b = tokens[j + 1]

            # cases where we want mult
            # x(, 2(, x x, 2 x, )x, )2
            if (
                (a.isdigit() and b.isalpha()) or
                (a.isalpha() and b.isalpha()) or
                (a in [")"] and (b.isalpha() or b.isdigit())) or
                ((a.isalpha() or a.isdigit()) and b in ["("])
            ):
                new_tokens.append("*")

    return new_tokens

In [9]:
def build_vocab(token_lists):
    # extra characters
    vocab = ["<PAD>", "<SOS>", "<EOS>", "<SEP>", "<UNK>"]

    for tokens in token_lists:
        for token in tokens:
            if token not in vocab:
                vocab.append(token)

    token_to_id = {token: i for i, token in enumerate(vocab)}
    id_to_token = {i: token for token, i in token_to_id.items()}

    return vocab, token_to_id, id_to_token

In [6]:
def encode(tokens, token_to_id):
    encoded = []

    for token in tokens:
        if token in token_to_id:
            encoded.append(token_to_id[token])
        else:
            encoded.append(token_to_id["<UNK>"])

    return encoded

In [7]:
def decode(ids, id_to_token):
    tokens = []

    for idx in ids:
        token = id_to_token[idx]

        if token == "<EOS>":
            break

        if token in ["<PAD>", "<SOS>"]:
            continue

        tokens.append(token)

    return tokens

In [10]:
df = train_df
full_token_lists = []

for inp, tgt in zip(df["input_text"], df["target_text"]):
    input_tokens = algebra_splitter(inp)
    target_tokens = algebra_splitter(tgt)

    full_tokens = ["<SOS>"] + input_tokens + ["<SEP>"] + target_tokens + ["<EOS>"]
    full_token_lists.append(full_tokens)

# build vocab
vocab, token_to_id, id_to_token = build_vocab(full_token_lists)

print("Number of equations:", len(full_token_lists))
print("Vocabulary size:", len(vocab))
print("First split example:", full_token_lists[0])


Number of equations: 37541
Vocabulary size: 115
First split example: ['<SOS>', '7', '*', 'x', '-', '8', '=', '-', '29', '->', '<SEP>', '7', '*', 'x', '=', '-', '21', '<EOS>']


In [15]:
class TargetOnlyNextTokenDataset(Dataset):
    def __init__(self, full_token_lists, token_to_id, context_length):
        self.data = []
        self.token_to_id = token_to_id
        self.context_length = context_length
        self.pad_id = token_to_id["<PAD>"]

        for tokens in full_token_lists:
            encoded = encode(tokens, token_to_id)

            sep_index = tokens.index("<SEP>")

            for i in range(sep_index + 1, len(encoded)):
                # train only on target-side tokens
                start = max(0, i - context_length)

                context = encoded[start:i]

                # left pad to fixed context length
                if len(context) < context_length:
                    context = [self.pad_id] * (context_length - len(context)) + context

                target = encoded[i]

                self.data.append((context, target))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        context, target = self.data[idx]
        return torch.tensor(context, dtype=torch.long), torch.tensor(target, dtype=torch.long)

In [11]:
class SeussModel(nn.Module):
    def __init__(self, vocab_size, context_length, embedding_dim, hidden_dim):
        super().__init__()
        self.context_length = context_length
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pipeline = nn.Sequential(
            nn.Linear(context_length * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, x):
        embedded = self.embedding(x)
        flattened = embedded.view(embedded.shape[0], -1)
        return self.pipeline(flattened)

In [12]:
# similar to previous model but with attention heads
class MathModel1(nn.Module):
    def __init__(self, vocab_size, context_length, embedding_dim, hidden_dim):
        super().__init__()

        self.context_length = context_length
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=4,
            batch_first=True # I could not figure out the error not having this setting was causing and used chatGPT for assistance.
        )

        self.norm1 = nn.LayerNorm(embedding_dim)

        self.net = nn.Sequential(
            nn.Linear(context_length * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, x):
        embedded = self.embedding(x)

        attn_out, _ = self.attention(
            embedded,
            embedded,
            embedded
        )

        out = self.norm1(embedded + attn_out)

        flattened = out.reshape(out.shape[0], -1)

        return self.net(flattened)

In [13]:
# full transformer block model
class MathModel2(nn.Module):
  #https://www.geeksforgeeks.org/deep-learning/transformer-using-pytorch/
  #https://docs.pytorch.org/docs/2.11/generated/torch.nn.TransformerEncoderLayer.html?utm_source=chatgpt.com

    def __init__(
        self,
        vocab_size,
        context_length,
        embedding_dim=64,
        hidden_dim=256,
        num_heads=4,
        num_layers=4,
        pad_id=0,
        dropout=0.1
    ):
        super().__init__()

        self.context_length = context_length
        self.pad_id = pad_id

        # token embeddings
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)

        # positional embeddings
        self.position_embedding = nn.Embedding(context_length, embedding_dim)

        # transformer block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True,
            activation="relu"
        )

        # stack multiple transformer blocks
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(embedding_dim)

        # predict next token from final context position
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):

        batch_size, seq_len = x.shape

        positions = torch.arange(seq_len, device=x.device)
        positions = positions.unsqueeze(0).expand(batch_size, seq_len)

        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(positions)

        out = token_emb + pos_emb

        # mask padding tokens so attention does not focus on <PAD>
        padding_mask = x == self.pad_id

        out = self.transformer(
            out,
            src_key_padding_mask=padding_mask
        )

        out = self.norm(out)

        # use the last token in the context to predict the next token
        last_token = out[:, -1, :]

        logits = self.output(last_token)

        return logits

In [16]:
context_length = 10

vocab, token_to_id, id_to_token = build_vocab(full_token_lists)

dataset = TargetOnlyNextTokenDataset(
    full_token_lists,
    token_to_id,
    context_length=context_length
)

loader = DataLoader(dataset, batch_size=3200, shuffle=True)

print("Training examples:", len(dataset))
print("Vocab size:", len(vocab))

Training examples: 292755
Vocab size: 115


In [ ]:
# model = SeussModel(
#     vocab_size=len(vocab),
#     context_length=context_length,
#     embedding_dim=64,
#     hidden_dim=128
# ).to(device)
# summary(model)

In [17]:
model = MathModel1(
    vocab_size=len(vocab),
    context_length=context_length,
    embedding_dim=64,
    hidden_dim=128
).to(device)
summary(model)


Layer (type:depth-idx)                             Param #
MathModel1                                         --
├─Embedding: 1-1                                   7,360
├─MultiheadAttention: 1-2                          12,480
│    └─NonDynamicallyQuantizableLinear: 2-1        4,160
├─LayerNorm: 1-3                                   128
├─Sequential: 1-4                                  --
│    └─Linear: 2-2                                 82,048
│    └─ReLU: 2-3                                   --
│    └─Linear: 2-4                                 16,512
│    └─ReLU: 2-5                                   --
│    └─Linear: 2-6                                 14,835
Total params: 137,523
Trainable params: 137,523
Non-trainable params: 0

In [ ]:
# model = MathModel2(
#     vocab_size=len(vocab),
#     context_length=context_length,
#     embedding_dim=64,
#     hidden_dim=128,
#     num_heads=4,
#     num_layers=3,
#     pad_id=token_to_id["<PAD>"],
#     dropout=0.1
# ).to(device)
# summary(model)


In [18]:
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

loss_history = []

for epoch in range(200):
    model.train()
    total_loss = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        opt.zero_grad()

        preds = model(X_batch)

        loss = loss_fn(preds, y_batch)
        loss.backward()
        opt.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    loss_history.append(avg_loss)

    print(f"Epoch {epoch+1}: loss = {avg_loss:.4f}")

KeyboardInterrupt: 

In [ ]:
with open("model_sm.pkl", "wb") as f:
    pickle.dump(model, f)